# 04. 장르 종합 분석 — 시장 규모 · 만족도 · 가격 · 성과 등급

**분석 목적:** 인디 개발사가 장르를 선택할 때 참고할 수 있도록, 장르별 시장 규모(리뷰 수)·유저 만족도(긍정률)·가격 포지셔닝·성과 등급 분포를 하나의 흐름으로 분석한다.

**사용 데이터:** `data/preprocessed/steam_indie_games_graded.csv` (리뷰 10개 이상, 2023~2025년, EA·F2P 제외, 성과 등급 컬럼 포함)

**분석 흐름:**
1. 데이터 로드 및 장르 explode
2. 장르별 게임 수
3. 장르별 리뷰 수 분포 (시장 규모)
4. 장르별 긍정률 분포 (유저 만족도)
5. 장르별 가격 분포 및 가격대 × 긍정률 히트맵
6. 장르별 성과 등급 분포 히트맵 (규모 · 만족도)
7. 장르별 포지셔닝 버블 차트 (규모 × 만족도)
8. 장르별 흥행 전환율
9. 출시 연도 × 장르 성과 추이
10. 종합 요약

---

> **분석 방법론 — 다중 장르 중복 집계**
>
> 실제 Steam 인디게임 대부분은 복합 장르를 가진다. 게임이 가진 모든 장르에 중복 집계하는 방식을 선택해 **"해당 장르 속성을 가진 게임군의 경향성"** 을 분석한다.
>
> | 한계 | 설명 |
> |------|------|
> | 중복 집계 | 동일 게임이 여러 장르에 포함되어 장르 간 독립성이 없다 |
> | 표본 과장 | 장르별 n수가 실제 고유 게임 수보다 많아 통계적 유의성이 과장될 수 있다 |
> | 조합 효과 미반영 | "Action+RPG 조합"의 성과와 "Action 단독"의 성과를 구분하지 않는다 |

> **성과 등급 분류 기준 (Steam 리뷰 수 × 긍정률 기준, 실제 매출과 무관)**
>
> | 등급명 | 기준 |
> |--------|------|
> | 대흥행 | 리뷰 500개↑ · 긍정률 80%↑ |
> | 상업적 성공 | 리뷰 500개↑ · 긍정률 70~79% |
> | 호불호 | 리뷰 500개↑ · 긍정률 70%↓ |
> | 숨겨진 명작 | 리뷰 50~499개 · 긍정률 80%↑ |
> | 평범 | 리뷰 50~499개 · 긍정률 70~79% |
> | 외면 | 리뷰 50~499개 · 긍정률 70%↓ |
> | 니치 (틈새) | 리뷰 10~49개 · 긍정률 80%↑ |
> | 미노출 | 리뷰 10~49개 · 긍정률 70~79% |
> | 미반응 | 리뷰 10~49개 · 긍정률 70%↓ |

## 0. 라이브러리 로드 및 공통 설정

In [15]:
import ast
import warnings

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

TARGET_GENRES = ['Action', 'Adventure', 'Casual', 'RPG', 'Simulation', 'Strategy', 'Sports', 'Racing']

PALETTE = [
    '#4C72B0', '#DD8452', '#55A868', '#C44E52',
    '#8172B2', '#937860', '#DA8BC3', '#8C8C8C'
]
COLOR_MAP = {g: PALETTE[i] for i, g in enumerate(TARGET_GENRES)}

GRADE_ORDER = [
    'high_high', 'high_mid', 'high_low',
    'mid_high',  'mid_mid',  'mid_low',
    'low_high',  'low_mid',  'low_low',
]
GRADE_LABEL = {
    'high_high': '대흥행',
    'high_mid' : '상업적 성공',
    'high_low' : '호불호',
    'mid_high' : '숨겨진 명작',
    'mid_mid'  : '평범',
    'mid_low'  : '외면',
    'low_high' : '니치 (틈새)',
    'low_mid'  : '미노출',
    'low_low'  : '미반응',
}
GRADE_LABEL_SUB = {
    'high_high': '리뷰 500개↑ · 긍정률 80%↑',
    'high_mid' : '리뷰 500개↑ · 긍정률 70~79%',
    'high_low' : '리뷰 500개↑ · 긍정률 70%↓',
    'mid_high' : '리뷰 50~499개 · 긍정률 80%↑',
    'mid_mid'  : '리뷰 50~499개 · 긍정률 70~79%',
    'mid_low'  : '리뷰 50~499개 · 긍정률 70%↓',
    'low_high' : '리뷰 10~49개 · 긍정률 80%↑',
    'low_mid'  : '리뷰 10~49개 · 긍정률 70~79%',
    'low_low'  : '리뷰 10~49개 · 긍정률 70%↓',
}

# 만족도 등급 그룹
HIGH_SAT   = ['high_high', 'mid_high', 'low_high']
MID_SAT    = ['high_mid',  'mid_mid',  'low_mid']
LOW_SAT    = ['high_low',  'mid_low',  'low_low']
HIGH_SCALE = ['high_high', 'high_mid', 'high_low']
LOW_SCALE  = ['low_high',  'low_mid',  'low_low']

## 1. 데이터 로드 및 장르 explode

In [16]:
games = pd.read_csv('../../../data/preprocessed/steam_indie_games_graded.csv')
games['release_year'] = pd.to_datetime(games['release_date']).dt.year

# positive_rate가 없으면 계산
if 'positive_rate' not in games.columns:
    games['positive_rate'] = games['positive'] / games['total_reviews'] * 100

games['genres_list'] = games['genres'].apply(lambda g: ast.literal_eval(g) if pd.notna(g) else [])
games['genres_filtered'] = games['genres_list'].apply(
    lambda gl: [g for g in gl if g in TARGET_GENRES]
)

games_with_genre = games[games['genres_filtered'].map(len) > 0].copy()
df = (
    games_with_genre
    .explode('genres_filtered')
    .rename(columns={'genres_filtered': 'genre'})
    .reset_index(drop=True)
)
df['grade_label'] = df['performance_grade'].map(GRADE_LABEL)

# 유료 게임 (가격 분석용)
df_paid = df[(df['price'] > 0) & (df['price'] <= 60)].copy()

print(f'원본 게임 수    : {len(games_with_genre):,}개')
print(f'explode 후 행 수: {len(df):,}행 (중복 포함)')
print()
print('장르별 집계 게임 수:')
print(df['genre'].value_counts().to_string())

원본 게임 수    : 8,730개
explode 후 행 수: 19,009행 (중복 포함)

장르별 집계 게임 수:
genre
Adventure     4496
Casual        3870
Action        3846
Simulation    2308
RPG           1967
Strategy      1916
Sports         323
Racing         283


## 2. 장르별 게임 수

In [17]:
genre_counts = df['genre'].value_counts().reset_index()
genre_counts.columns = ['genre', 'count']

fig = px.bar(
    genre_counts,
    x='genre', y='count',
    color='genre',
    color_discrete_map=COLOR_MAP,
    text='count',
    title='장르별 게임 수 (다중 장르 중복 집계)',
    labels={'genre': '장르', 'count': '게임 수 (중복 포함)'},
)
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.update_layout(showlegend=False, xaxis_categoryorder='total descending')
fig.show()

**해석:** Adventure·Action·Casual이 게임 수 기준 상위 3개 장르다. RPG·Simulation·Strategy는 단독 장르보다 Action·Adventure와 조합되는 경우가 많아 이 방식에서도 상당한 수를 차지한다.

## 3. 장르별 리뷰 수 분포 (시장 규모)

In [18]:
genre_order_reviews = (
    df.groupby('genre')['total_reviews']
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

fig = px.box(
    df,
    x='genre', y='total_reviews',
    category_orders={'genre': genre_order_reviews},
    color='genre',
    color_discrete_map=COLOR_MAP,
    points=False,
    title='장르별 총 리뷰 수 분포 (다중 장르 방식)',
    labels={'genre': '장르', 'total_reviews': '총 리뷰 수 (log scale)'},
)
fig.update_layout(showlegend=False, yaxis_type='log')
fig.show()

print('장르별 리뷰 수 통계:')
display(
    df.groupby('genre')['total_reviews']
    .agg(['count', 'median', 'mean', 'std'])
    .round(1)
    .sort_values('median', ascending=False)
)

장르별 리뷰 수 통계:


,count,median,mean,std
genre,,,,
RPG,1967,66.0,1264.8,10985.7
Simulation,2308,54.0,1303.6,12418.3
Strategy,1916,51.0,1073.1,11016.9
Adventure,4496,41.0,915.0,9811.3
Sports,323,37.0,264.4,1685.7
Action,3846,36.0,1045.5,10386.7
Casual,3870,35.0,461.1,3832.5
Racing,283,27.0,603.5,5651.0


**해석:** 리뷰 수 중앙값이 높은 장르(RPG·Simulation·Strategy)는 게임당 플레이타임이 길고 몰입도 높은 유저층을 보유하는 경향이 있어 더 많은 유저 참여를 이끌어낸다. Casual·Racing은 중앙값이 낮아 개별 게임의 리뷰 집중도가 상대적으로 낮다.

## 4. 장르별 긍정률 분포 (유저 만족도)

In [19]:
genre_order_pos = (
    df.groupby('genre')['positive_rate']
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

fig = px.box(
    df,
    x='genre', y='positive_rate',
    category_orders={'genre': genre_order_pos},
    color='genre',
    color_discrete_map=COLOR_MAP,
    points=False,
    title='장르별 긍정률 분포 (다중 장르 방식)',
    labels={'genre': '장르', 'positive_rate': '긍정률 (%)'},
)
fig.add_hline(y=80, line_dash='dash', line_color='red',
              annotation_text='80% (Very Positive)', annotation_position='top right')
fig.update_layout(showlegend=False, yaxis_range=[0, 110])
fig.show()

print('장르별 긍정률 통계:')
display(
    df.groupby('genre')['positive_rate']
    .agg(['count', 'median', 'mean', 'std'])
    .round(2)
    .sort_values('median', ascending=False)
)

장르별 긍정률 통계:


,count,median,mean,std
genre,,,,
Casual,3870,90.00,85.58,14.93
Action,3846,88.24,84.14,15.31
Adventure,4496,88.00,83.95,14.94
Racing,283,87.70,82.98,16.23
Strategy,1916,86.67,83.34,14.52
RPG,1967,86.36,82.87,14.92
Sports,323,86.00,83.13,14.78
Simulation,2308,84.00,79.99,16.51


**해석:** 모든 장르의 긍정률 중앙값이 80% 이상으로 Steam 'Very Positive' 수준을 유지한다. Casual이 중앙값 기준 가장 높고 Simulation이 가장 낮다. Simulation의 표준편차가 가장 커서 품질 편차가 크다는 점에 주목할 만하다.

## 5. 장르별 가격 분포 및 가격대 × 긍정률 히트맵

In [20]:
genre_order_price = (
    df_paid.groupby('genre')['price']
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

fig = px.box(
    df_paid,
    x='genre', y='price',
    category_orders={'genre': genre_order_price},
    color='genre',
    color_discrete_map=COLOR_MAP,
    points=False,
    title='장르별 출시 가격 분포 (유료 게임, 다중 장르 방식)',
    labels={'genre': '장르', 'price': '가격 (USD)'},
)
fig.update_layout(showlegend=False, yaxis_range=[0, 65])
fig.update_yaxes(tickprefix='$')
fig.show()

print('장르별 가격 통계:')
display(
    df_paid.groupby('genre')['price']
    .agg(['count', 'median', 'mean', 'std'])
    .round(2)
    .sort_values('median', ascending=False)
)

장르별 가격 통계:


,count,median,mean,std
genre,,,,
RPG,1958,8.99,10.34,7.67
Strategy,1904,7.99,10.01,7.52
Action,3831,6.99,8.90,7.15
Adventure,4481,6.99,9.04,7.30
Simulation,2291,6.99,9.18,7.65
Sports,313,6.99,9.85,9.05
Casual,3852,4.99,7.30,6.22
Racing,275,4.99,8.06,6.99


**해석:** RPG·Strategy가 가격 중앙값이 높고 Casual·Racing이 낮다. 신규 인디 개발사라면 해당 장르의 중앙값 ±20% 범위를 출시 가격의 기준점으로 삼는 것을 권장한다.

In [21]:
price_bins   = [0, 5, 10, 15, 20, 30, 50, float('inf')]
price_labels = ['~$5', '$5~10', '$10~15', '$15~20', '$20~30', '$30~50', '$50↑']
df_paid = df_paid.copy()
df_paid['price_range'] = pd.cut(df_paid['price'], bins=price_bins, labels=price_labels, right=True)

MIN_CELL = 5
heatmap_data = (
    df_paid
    .groupby(['genre', 'price_range'], observed=True)
    .agg(avg_pos=('positive_rate', 'mean'), count=('appid', 'nunique'))
    .reset_index()
)
heatmap_data.loc[heatmap_data['count'] < MIN_CELL, 'avg_pos'] = float('nan')
pivot = heatmap_data.pivot(index='genre', columns='price_range', values='avg_pos').round(1)

fig = px.imshow(
    pivot,
    text_auto='.1f',
    color_continuous_scale='RdYlGn',
    zmin=70, zmax=95,
    title=f'장르 × 가격대별 평균 긍정률<br><sub>게임 수 {MIN_CELL}개 미만 셀 제외 / 다중 장르 중복 집계</sub>',
    labels={'x': '가격대', 'y': '장르', 'color': '평균 긍정률 (%)'},
    aspect='auto',
)
fig.update_layout(height=400)
fig.show()

**해석:**
- **저가 구간(~$5):** Casual·Action·Adventure에서 게임 수가 집중되고 긍정률도 유지된다.
- **중가 구간($10~$20):** 대부분의 장르에서 게임 수가 줄어들지만 리뷰 수(규모)는 오히려 늘어나는 경향 — 구매 가격이 높을수록 실제 플레이어가 리뷰를 남기는 비율이 높다.
- **고가 구간($30↑):** 게임 수가 적어 셀이 비는 장르가 많다. 긍정률이 유지되더라도 표본이 소수이므로 해석에 주의가 필요하다.

## 6. 장르별 성과 등급 분포 히트맵 (규모 · 만족도)

9개 등급을 한 번에 보면 두 축(규모·만족도)이 섞여 해석이 어렵다. 규모(`scale_grade`)와 만족도(`satisfaction_grade`)를 분리해 각각 히트맵으로 시각화한다.

In [22]:
SCALE_ORDER = ['high', 'mid', 'low']
SCALE_LABEL = {'high': 'high (리뷰 500개↑)', 'mid': 'mid (리뷰 50~499개)', 'low': 'low (리뷰 10~49개)'}
SAT_ORDER   = ['high', 'mid', 'low']
SAT_LABEL   = {'high': 'high (긍정률 80%↑)', 'mid': 'mid (긍정률 70~79%)', 'low': 'low (긍정률 70%↓)'}

scale_cross = pd.crosstab(df['genre'], df['scale_grade'], normalize='index') * 100
scale_cross = scale_cross.reindex(columns=SCALE_ORDER)
scale_cross.columns = [SCALE_LABEL[c] for c in SCALE_ORDER]
scale_cross_sorted = scale_cross.loc[
    scale_cross[SCALE_LABEL['high']].sort_values(ascending=False).index
]

sat_cross = pd.crosstab(df['genre'], df['satisfaction_grade'], normalize='index') * 100
sat_cross = sat_cross.reindex(columns=SAT_ORDER)
sat_cross.columns = [SAT_LABEL[c] for c in SAT_ORDER]
sat_cross_sorted = sat_cross.loc[
    sat_cross[SAT_LABEL['high']].sort_values(ascending=False).index
]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        '장르별 리뷰 규모 분포 (%, 행 합계 = 100)',
        '장르별 만족도 분포 (%, 행 합계 = 100)',
    ],
    horizontal_spacing=0.12,
)
fig.add_trace(
    go.Heatmap(
        z=scale_cross_sorted.round(1).values,
        x=scale_cross_sorted.columns.tolist(),
        y=scale_cross_sorted.index.tolist(),
        colorscale='Blues',
        text=scale_cross_sorted.round(1).values,
        texttemplate='%{text}',
        showscale=True,
        colorbar=dict(x=0.44, title='%'),
    ),
    row=1, col=1,
)
fig.add_trace(
    go.Heatmap(
        z=sat_cross_sorted.round(1).values,
        x=sat_cross_sorted.columns.tolist(),
        y=sat_cross_sorted.index.tolist(),
        colorscale='Greens',
        text=sat_cross_sorted.round(1).values,
        texttemplate='%{text}',
        showscale=True,
        colorbar=dict(x=1.0, title='%'),
    ),
    row=1, col=2,
)
fig.update_layout(
    title='장르별 리뷰 규모·만족도 분포 히트맵<br><sub>다중 장르 중복 집계 / high 비율 내림차순 정렬 / 실제 매출과 무관</sub>',
    height=420,
    width=1100,
)
fig.update_xaxes(tickangle=-15)
fig.show()

**해석:**
- **규모 히트맵 — `high` 비율이 높은 장르:** 리뷰 500개 이상을 확보하는 게임이 많다. 시장 규모가 크고 경쟁도 치열한 장르다.
- **규모 히트맵 — `low` 비율이 높은 장르:** 대부분의 게임이 리뷰를 많이 받지 못한다. 노출 확보가 어려운 장르거나 틈새 시장 성격이 강하다.
- **만족도 히트맵 — `high` 비율이 높은 장르:** 긍정률 80% 이상을 달성하는 게임이 많다. 유저 기대치에 부응하기 상대적으로 쉬운 장르다.
- **만족도 히트맵 — `low` 비율이 높은 장르:** 긍정률 70% 미만 게임이 많다. 유저 기대치가 높거나 품질 편차가 큰 장르다.

## 7. 장르별 포지셔닝 버블 차트 (규모 × 만족도)

x축: 중앙값 리뷰 수, y축: 중앙값 긍정률, 버블 크기: 게임 수 — 4분면으로 장르의 시장 포지션을 직관적으로 파악한다.

In [23]:
genre_agg = df.groupby('genre').agg(
    리뷰수_중앙값=('total_reviews', 'median'),
    긍정률_중앙값=('positive_rate', 'median'),
    게임수=('appid', 'count'),
    대흥행_비율=('performance_grade', lambda x: (x == 'high_high').mean() * 100),
).reset_index().round(2)

review_mid = genre_agg['리뷰수_중앙값'].median()
pos_mid    = genre_agg['긍정률_중앙값'].median()

fig = px.scatter(
    genre_agg,
    x='리뷰수_중앙값',
    y='긍정률_중앙값',
    size='게임수',
    color='genre',
    color_discrete_map=COLOR_MAP,
    text='genre',
    size_max=60,
    hover_data={'대흥행_비율': ':.1f', '게임수': True},
    title='장르별 포지셔닝: 시장 규모 × 유저 만족도<br><sub>버블 크기 = 게임 수 / 중앙값 기준 / 다중 장르 중복 집계</sub>',
    labels={'리뷰수_중앙값': '중앙값 리뷰 수 (규모)', '긍정률_중앙값': '중앙값 긍정률 (%)'},
)
fig.update_traces(textposition='top center', marker=dict(opacity=0.8))
fig.add_vline(x=review_mid, line_dash='dot', line_color='gray', opacity=0.6,
              annotation_text='규모 중앙값', annotation_position='top right')
fig.add_hline(y=pos_mid, line_dash='dot', line_color='gray', opacity=0.6,
              annotation_text='만족도 중앙값', annotation_position='top right')
fig.update_layout(showlegend=False, height=520,
                  yaxis=dict(range=[80, 93]))
fig.show()

print('\n장르별 집계 요약:')
display(genre_agg.set_index('genre').sort_values('리뷰수_중앙값', ascending=False))


장르별 집계 요약:


,리뷰수_중앙값,긍정률_중앙값,게임수,대흥행_비율
genre,,,,
RPG,66.0,86.36,1967,13.01
Simulation,54.0,84.00,2308,12.39
Strategy,51.0,86.67,1916,11.53
Adventure,41.0,88.00,4496,9.27
Sports,37.0,86.00,323,4.33
Action,36.0,88.24,3846,9.39
Casual,35.0,90.00,3870,7.73
Racing,27.0,87.70,283,6.71


**4분면 해석:**

| 위치 | 특성 | 전략 제안 |
|------|------|----------|
| 우상단 (규모↑ 만족도↑) | 주류 시장, 높은 경쟁 | 품질 차별화 필수, 성공 시 파급력 큼 |
| 좌상단 (규모↓ 만족도↑) | 틈새 시장, 팬층 두터움 | 커뮤니티 마케팅으로 규모 확장 가능 |
| 우하단 (규모↑ 만족도↓) | 기대치 대비 실망 큼 | 출시 전 QA·완성도 집중 |
| 좌하단 (규모↓ 만족도↓) | 시장·만족도 모두 약함 | 신중한 장르 선택 또는 세부 차별화 필요 |

## 8. 장르별 흥행 전환율

각 장르에서 만족도 높은 구간(대흥행 + 숨겨진 명작 + 니치)과 미반응 구간의 비율을 비교한다.

In [24]:
def classify_sat(grade):
    if grade in HIGH_SAT:
        return '높은 만족도 (≥80%)'
    elif grade in MID_SAT:
        return '보통 만족도 (70~79%)'
    return '낮은 만족도 (<70%)'

df['sat_group'] = df['performance_grade'].apply(classify_sat)
SAT_GROUP_ORDER  = ['높은 만족도 (≥80%)', '보통 만족도 (70~79%)', '낮은 만족도 (<70%)']
SAT_GROUP_COLORS = {
    '높은 만족도 (≥80%)': '#2d6a4f',
    '보통 만족도 (70~79%)': '#adb5bd',
    '낮은 만족도 (<70%)': '#C44E52',
}

sat_pct = (
    df.groupby(['genre', 'sat_group'])
    .size()
    .reset_index(name='count')
)
sat_pct['pct'] = sat_pct.groupby('genre')['count'].transform(lambda x: x / x.sum() * 100)

genre_high_sat_order = (
    sat_pct[sat_pct['sat_group'] == '높은 만족도 (≥80%)']
    .set_index('genre')['pct']
    .sort_values(ascending=False)
    .index.tolist()
)

fig = px.bar(
    sat_pct,
    x='genre', y='pct',
    color='sat_group',
    color_discrete_map=SAT_GROUP_COLORS,
    category_orders={'genre': genre_high_sat_order, 'sat_group': SAT_GROUP_ORDER},
    text='pct',
    barmode='stack',
    title='장르별 만족도 구간 비율 (스택 바)<br><sub>다중 장르 중복 집계</sub>',
    labels={'genre': '장르', 'pct': '비율 (%)', 'sat_group': '만족도 구간'},
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='inside', textfont_size=11)
fig.update_layout(height=450, yaxis_range=[0, 105])
fig.show()

In [25]:
conversion = (
    df.groupby('genre')['performance_grade']
    .agg(
        대흥행_비율=lambda x: (x == 'high_high').mean() * 100,
        만족도높음_비율=lambda x: x.isin(HIGH_SAT).mean() * 100,
        규모높음_비율=lambda x: x.isin(HIGH_SCALE).mean() * 100,
        미반응_비율=lambda x: (x == 'low_low').mean() * 100,
    )
    .round(1)
    .sort_values('대흥행_비율', ascending=False)
)
conversion.columns = ['대흥행_비율(%)', '만족도높음_비율(%)', '규모높음_비율(%)', '미반응_비율(%)']

print('장르별 흥행 지표 요약:')
display(conversion)

장르별 흥행 지표 요약:


,대흥행_비율(%),만족도높음_비율(%),규모높음_비율(%),미반응_비율(%)
genre,,,,
RPG,13.0,67.4,17.8,8.8
Simulation,12.4,60.1,16.1,13.1
Strategy,11.5,67.5,16.1,8.1
Action,9.4,70.1,12.5,9.7
Adventure,9.3,69.6,12.2,10.1
Casual,7.7,74.6,9.4,9.1
Racing,6.7,66.1,8.1,13.1
Sports,4.3,67.5,5.9,12.1


**해석:**
- **만족도 높음 비율:** 해당 장르 속성을 가진 게임 중 긍정률 80% 이상을 달성하는 비율 — 품질 달성 가능성을 나타낸다.
- **대흥행 비율:** 리뷰 500개 이상 + 긍정률 80% 이상을 동시에 달성하는 비율 — 장르의 진정한 흥행 전환율이다.
- **미반응 비율:** 리뷰 49개 이하 + 긍정률 70% 미만인 비율 — 장르 진입 리스크를 나타낸다.

인디 개발사라면 **대흥행 비율이 높고 미반응 비율이 낮은 장르**를 우선 고려할 수 있다.

## 9. 출시 연도 × 장르 성과 추이 (2023~2025)

In [26]:
trend = (
    df.groupby(['release_year', 'genre'])['performance_grade']
    .agg(
        대흥행_비율=lambda x: (x == 'high_high').mean() * 100,
        미반응_비율=lambda x: (x == 'low_low').mean() * 100,
        게임수=lambda x: len(x),
    )
    .round(2)
    .reset_index()
)
trend = trend[trend['게임수'] >= 20]  # 연도별 충분한 데이터가 있는 장르만

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('장르별 대흥행 비율 추이', '장르별 미반응 비율 추이'),
)
for genre in TARGET_GENRES:
    g = trend[trend['genre'] == genre]
    if len(g) < 2:
        continue
    color = COLOR_MAP[genre]
    fig.add_trace(go.Scatter(
        x=g['release_year'], y=g['대흥행_비율'],
        mode='lines+markers', name=genre,
        line=dict(color=color), marker=dict(size=7),
        legendgroup=genre,
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=g['release_year'], y=g['미반응_비율'],
        mode='lines+markers', name=genre,
        line=dict(color=color, dash='dot'), marker=dict(size=7),
        legendgroup=genre, showlegend=False,
    ), row=1, col=2)

fig.update_layout(
    title='출시 연도 × 장르 성과 추이 (2023~2025)<br><sub>대흥행: 리뷰 500개↑ + 긍정률 80%↑ / 미반응: 리뷰 49개↓ + 긍정률 70%↓</sub>',
    height=480,
    legend=dict(title='장르'),
)
fig.update_xaxes(tickvals=[2023, 2024, 2025], ticktext=['2023', '2024', '2025'])
fig.update_yaxes(title_text='비율 (%)', row=1, col=1)
fig.update_yaxes(title_text='비율 (%)', row=1, col=2)
fig.show()

In [27]:
trend_pivot = trend.pivot_table(
    index='genre', columns='release_year',
    values='대흥행_비율', aggfunc='first'
).round(1)
trend_pivot.columns.name = '출시연도 (대흥행 비율 %)'

print('연도별 대흥행 비율 (%):')
display(trend_pivot.sort_values(2025, ascending=False, na_position='last'))

연도별 대흥행 비율 (%):


출시연도 (대흥행 비율 %),2023,2024,2025
genre,,,
Simulation,10.9,12.6,14.0
Racing,5.4,5.6,12.5
RPG,13.0,13.3,12.4
Strategy,10.9,12.3,11.1
Action,9.2,9.8,8.9
Adventure,9.7,9.8,7.5
Casual,8.4,7.6,6.9
Sports,4.5,4.6,3.3


**해석:**
- 대흥행 비율이 **증가하는 장르**는 시장이 성장 중이거나 흥행작 비율이 높아지는 추세다.
- 대흥행 비율이 **감소하고 미반응 비율이 증가하는 장르**는 공급 과잉 또는 시장 포화 신호일 수 있다.
- 단, 2025년은 데이터 수집 시점 기준으로 출시 후 충분한 시간이 경과하지 않은 게임이 포함될 수 있어 비율이 낮게 나타날 수 있다.

## 10. 종합 요약

In [28]:
summary = df.groupby('genre').agg(
    게임수=('appid', 'count'),
    리뷰수_중앙값=('total_reviews', 'median'),
    긍정률_중앙값=('positive_rate', 'median'),
    긍정률_std=('positive_rate', 'std'),
    대흥행_비율=('performance_grade', lambda x: (x == 'high_high').mean() * 100),
    미반응_비율=('performance_grade', lambda x: (x == 'low_low').mean() * 100),
).round(2)

price_med = df_paid.groupby('genre')['price'].median().rename('가격_중앙값_USD')
summary = summary.join(price_med)
summary['리뷰수_중앙값'] = summary['리뷰수_중앙값'].astype(int)
summary = summary.sort_values('대흥행_비율', ascending=False)

print('장르별 종합 지표:')
display(summary)

장르별 종합 지표:


,게임수,리뷰수_중앙값,긍정률_중앙값,긍정률_std,대흥행_비율,미반응_비율,가격_중앙값_USD
genre,,,,,,,
RPG,1967,66,86.36,14.92,13.01,8.80,8.99
Simulation,2308,54,84.00,16.51,12.39,13.08,6.99
Strategy,1916,51,86.67,14.52,11.53,8.14,7.99
Action,3846,36,88.24,15.31,9.39,9.70,6.99
Adventure,4496,41,88.00,14.94,9.27,10.05,6.99
Casual,3870,35,90.00,14.93,7.73,9.10,4.99
Racing,283,27,87.70,16.23,6.71,13.07,4.99
Sports,323,37,86.00,14.78,4.33,12.07,6.99


### 인디 개발사를 위한 장르 선택 가이드

| 분석 | 핵심 인사이트 |
|------|-------------|
| 리뷰 수 분포 | 장르별 시장 규모(노출·참여) 차이 확인 |
| 긍정률 분포 | 장르별 만족도 수준 및 품질 편차 파악 |
| 가격 히트맵 | 장르 × 가격대별 긍정률 — 적정 출시 가격 기준점 제공 |
| 성과 등급 히트맵 | 규모·만족도 축별 장르 특성 파악 |
| 포지셔닝 버블 | 경쟁 강도와 팬층 특성의 장르별 차이 |
| 흥행 전환율 | 장르별 대흥행·미반응 비율 — 진입 기대값과 리스크 수치화 |
| 연도별 추이 | 장르별 시장 성장·포화 방향 — 타이밍 전략 판단 근거 |

| 상황 | 추천 장르 선택 기준 |
|------|-------------------|
| 첫 출시, 리소스 제한 | 만족도높음 비율↑ + 미반응 비율↓ 장르 → 실패 리스크 최소화 |
| 시장 규모 우선 | 리뷰수 중앙값↑ 장르 → 단, 품질 차별화 필수 |
| 틈새 공략 | 버블 차트 좌상단 장르 → 커뮤니티 마케팅 병행 |
| 트렌드 타기 | 최근 2년 대흥행 비율 상승 장르 → 출시 타이밍 전략과 연계 |